#Crypto Laundering Forensics Lab

An educational toolkit for tracing stolen crypto with public blockchain data and comparing laundering patterns. Case studies: North Korea's TraderTraitor exchange thefts (Bybit 2025, Binance 2026)

| 단계 | 내용 | API 키 |
| --- | --- | --- |
| 0 | Create lab files (run the 3 cells below) | Not needed |
| 1 | Compare two laundering patterns on synthetic data | Not needed  |
| 2 | Read the metrics and report | Not needed |
| 3  (optional) | Trace real public addresses | Free Etherscan key |

**How to run**: Run the cells from top to bottom (▶ or `Shift+Enter`). No installation needed.

> ⚠️ Read-only: do not send funds to or interact with any listed address.실습은 공개 데이터를 **조회만** 합니다. 공개된 주소로 송금하거나 상호작용하지 마세요.


## 0. 실습 파일 만들기

아래 셀 3개를 실행하면 런타임에 스크립트 파일이 생깁니다. 코드를 읽어 보고 싶으면 셀을 펼쳐 보세요.

- `make_sample.py`: API 없이 쓰는 합성 세탁 사례 2개 생성
- `trace_flows.py`: 실제 공개 주소에서 출발해 자금 흐름 추적 (Etherscan API)
- `analyze.py`: 세탁 지표 계산과 사례 비교 리포트 생성

> Colab 런타임이 재시작되면 파일이 지워집니다. 그때는 이 셀들부터 다시 실행하세요.

In [ ]:
%%writefile trace_flows.py
#!/usr/bin/env python3
"""
trace_flows.py - 공개된 시드 주소(예: FBI 공개 목록)에서 출발해
이더리움 자금 흐름을 N-hop까지 추적하고, 출구(브릿지/스왑/거래소 등
컨트랙트)로 보이는 지점을 표시한다.

데이터 출처: Etherscan API V2 (무료 키: https://etherscan.io/myapikey)

사용법:
  export ETHERSCAN_API_KEY=키
  python3 trace_flows.py addresses.txt --since-block 23690000 --hops 3

  Colab: 왼쪽 열쇠(🔑) 아이콘 > 보안 비밀에 ETHERSCAN_API_KEY 를 등록하면
         키를 따로 넘기지 않아도 자동으로 읽는다.
         !python trace_flows.py addresses.txt --hops 2

입력:
  addresses.txt   한 줄에 주소 하나 (# 주석 가능)
  --labels FILE   (선택) address,label CSV. 알려진 거래소/브릿지 주소 등

출력 (out/ 폴더):
  nodes.csv       주소, hop, EOA/컨트랙트, 라벨, 유입/유출 합계
  edges.csv       송금 내역 (from, to, 자산, 수량, tx, 시각, 호출 함수)
  exits.csv       자금이 흘러든 컨트랙트(=출구 후보) 유입액 순위
  graph.html      브라우저로 여는 인터랙티브 흐름 그래프

주의: 공개 블록체인 데이터 조회만 수행한다. 이 주소들로 송금하거나
상호작용하지 말 것. 결과는 단서일 뿐이며, 의미 있는 발견은
tips.fbi.gov / ic3.gov 에 근거(tx 해시)와 함께 제보할 것.
"""
import argparse
import csv
import json
import os
import sys
import time
import urllib.parse
import urllib.request
from collections import defaultdict, deque
from datetime import datetime, timezone

API = "https://api.etherscan.io/v2/api"
CHAIN_ID = 1  # Ethereum mainnet

# 추적할 토큰과 최소 수량 (이보다 작은 이동은 무시)
TRACK_TOKENS = {
    "WETH": 1.0, "stETH": 1.0, "wstETH": 1.0, "cbETH": 1.0, "rETH": 1.0,
    "USDT": 10_000, "USDC": 10_000, "DAI": 10_000,
    "WBTC": 0.05, "cbBTC": 0.05,
}

# 호출 함수명에 이런 단어가 있으면 브릿지/스왑/믹서 출구 후보로 표시
EXIT_HINTS = [
    "bridge", "swap", "transferout", "outbound", "deposit", "relay",
    "send", "lock", "exactinput", "multicall", "execute", "fill",
]


class Etherscan:
    def __init__(self, key, rps=3.0):
        self.key = key
        self.min_gap = 1.0 / rps
        self.last = 0.0
        self.calls = 0
        self.ok = 0          # 성공한 호출 수 (키가 유효한지 판단에 사용)

    def get(self, **params):
        params.update(chainid=CHAIN_ID, apikey=self.key)
        url = API + "?" + urllib.parse.urlencode(params)
        for attempt in range(6):
            wait = self.min_gap - (time.time() - self.last)
            if wait > 0:
                time.sleep(wait)
            self.last = time.time()
            self.calls += 1
            try:
                with urllib.request.urlopen(url, timeout=30) as r:
                    data = json.loads(r.read().decode())
            except Exception as e:
                print(f"  ! 요청 오류({e}), 재시도", file=sys.stderr)
                time.sleep(2 * (attempt + 1))
                continue
            res = data.get("result")
            if isinstance(res, str) and "rate limit" in res.lower():
                time.sleep(1.5 * (attempt + 1))
                continue
            if data.get("status") == "0":
                msg = str(data.get("message", "")) + str(res)
                if "No transactions found" in msg or "No records found" in msg:
                    self.ok += 1
                    return []
                if ("Invalid API Key" in msg or "Missing" in msg) and self.ok > 0:
                    # 앞서 성공한 키가 갑자기 거부되면 대개 일시적 제한이다. 쉬었다가 재시도
                    print(f"  ! 일시적 키 거부({msg[:40]}), {5 * (attempt + 1)}초 후 재시도",
                          file=sys.stderr)
                    time.sleep(5 * (attempt + 1))
                    continue
                if "Invalid API Key" in msg or "Missing" in msg:
                    sys.exit(f"API 키 오류: {msg}\n"
                             "  - etherscan.io/myapikey 에서 만든 키인지 확인하세요 (BscScan 등 다른 사이트 키는 안 됩니다).\n"
                             "  - 보안 비밀 값에 공백·따옴표·줄바꿈 없이 키만 들어갔는지 확인하세요.\n"
                             "  - 방금 만든 키라면 몇 분 뒤 다시 시도하세요.")
                print(f"  ! API 응답: {msg[:120]}", file=sys.stderr)
                return []
            self.ok += 1
            return res
        print("  ! 재시도 한도 초과, 건너뜀", file=sys.stderr)
        return []

    def txlist(self, addr, start):
        return self.get(module="account", action="txlist", address=addr,
                        startblock=start, endblock=99999999,
                        page=1, offset=1000, sort="asc")

    def internal(self, addr, start):
        return self.get(module="account", action="txlistinternal", address=addr,
                        startblock=start, endblock=99999999,
                        page=1, offset=1000, sort="asc")

    def tokentx(self, addr, start):
        return self.get(module="account", action="tokentx", address=addr,
                        startblock=start, endblock=99999999,
                        page=1, offset=1000, sort="asc")

    def is_contract(self, addr):
        res = self.get(module="proxy", action="eth_getCode",
                       address=addr, tag="latest")
        return isinstance(res, str) and res not in ("0x", "0x0", "")


def get_api_key(cli_key=None):
    """--api-key > 환경변수 > Colab 보안 비밀 순으로 API 키를 찾는다."""
    clean = lambda k: (k or "").strip().strip("'\"").strip() or None  # 복사할 때 섞인 공백·따옴표 제거
    if cli_key:
        return clean(cli_key)
    if os.environ.get("ETHERSCAN_API_KEY"):
        return clean(os.environ["ETHERSCAN_API_KEY"])
    try:
        from google.colab import userdata  # Colab에서만 존재
        return clean(userdata.get("ETHERSCAN_API_KEY"))
    except Exception:
        return None


def ts(t):
    try:
        return datetime.fromtimestamp(int(t), tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return ""


def fn_name(tx):
    f = (tx.get("functionName") or "").split("(")[0]
    return f


def outgoing(es, addr, start, min_eth):
    """addr에서 나간 의미 있는 이동 목록"""
    edges = []
    for tx in es.txlist(addr, start):
        if tx.get("from", "").lower() != addr or tx.get("isError") == "1":
            continue
        to = (tx.get("to") or "").lower()
        if not to:
            continue
        val = int(tx.get("value", "0")) / 1e18
        f = fn_name(tx)
        # ETH 이동이거나, 값이 없어도 컨트랙트 호출(토큰 승인/브릿지 호출) 기록
        if val >= min_eth or (val > 0 and f):
            edges.append(dict(src=addr, dst=to, asset="ETH", amount=val,
                              tx=tx["hash"], time=ts(tx.get("timeStamp")), fn=f))
    for tx in es.internal(addr, start):
        if tx.get("from", "").lower() != addr or tx.get("isError") == "1":
            continue
        to = (tx.get("to") or "").lower()
        val = int(tx.get("value", "0")) / 1e18
        if to and val >= min_eth:
            edges.append(dict(src=addr, dst=to, asset="ETH(internal)", amount=val,
                              tx=tx["hash"], time=ts(tx.get("timeStamp")), fn=""))
    for tx in es.tokentx(addr, start):
        if tx.get("from", "").lower() != addr:
            continue
        sym = tx.get("tokenSymbol", "")
        if sym not in TRACK_TOKENS:
            continue
        dec = int(tx.get("tokenDecimal") or 18)
        amt = int(tx.get("value", "0")) / (10 ** dec)
        if amt >= TRACK_TOKENS[sym]:
            edges.append(dict(src=addr, dst=tx["to"].lower(), asset=sym, amount=amt,
                              tx=tx["hash"], time=ts(tx.get("timeStamp")), fn=""))
    return edges


def eth_equiv(e):
    """팬아웃 정렬용 대략적 크기 (ETH 계열만 비교, 나머지는 보조)"""
    if e["asset"].startswith("ETH") or e["asset"] in ("WETH", "stETH", "wstETH", "cbETH", "rETH"):
        return e["amount"]
    if e["asset"] in ("WBTC", "cbBTC"):
        return e["amount"] * 30
    return e["amount"] / 3000  # 스테이블코인 대략 환산


def load_list(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.split("#")[0].strip().strip(",")
            if line.lower().startswith("0x") and len(line) == 42:
                out.append(line.lower())
    return list(dict.fromkeys(out))


def load_labels(path):
    labels = {}
    if not path:
        return labels
    with open(path, encoding="utf-8") as f:
        for row in csv.reader(f):
            if len(row) >= 2 and row[0].lower().startswith("0x"):
                labels[row[0].strip().lower()] = row[1].strip()
    return labels


def main(argv=None):
    ap = argparse.ArgumentParser(description="공개 시드 주소 기반 ETH 자금 흐름 추적")
    ap.add_argument("addresses", help="시드 주소 파일 (한 줄에 하나)")
    ap.add_argument("--api-key", default=None, help="생략 시 환경변수/Colab 보안 비밀 사용")
    ap.add_argument("--since-block", type=int, default=0,
                    help="이 블록 이후 거래만 조회 (해킹 발생 블록 권장)")
    ap.add_argument("--hops", type=int, default=3, help="추적 깊이")
    ap.add_argument("--min-eth", type=float, default=1.0, help="최소 ETH 이동량")
    ap.add_argument("--max-children", type=int, default=15,
                    help="노드당 따라갈 최대 수신 주소 수 (큰 금액 순)")
    ap.add_argument("--max-nodes", type=int, default=600,
                    help="시드 외에 새로 추적할 최대 주소 수")
    ap.add_argument("--labels", help="address,label CSV (선택)")
    ap.add_argument("--rps", type=float, default=3.0, help="초당 API 호출 (무료 키는 3 권장)")
    ap.add_argument("--out", default="out")
    a = ap.parse_args(argv)

    a.api_key = get_api_key(a.api_key)
    if not a.api_key:
        sys.exit("API 키가 없습니다. --api-key, 환경변수 ETHERSCAN_API_KEY, "
                 "또는 Colab 보안 비밀(ETHERSCAN_API_KEY) 중 하나로 넘겨 주세요.")

    es = Etherscan(a.api_key, a.rps)
    seeds = load_list(a.addresses)
    if not seeds:
        sys.exit("시드 주소를 찾지 못했습니다.")
    labels = load_labels(a.labels)
    for s in seeds:
        labels.setdefault(s, "FBI seed")

    hop = {s: 0 for s in seeds}
    kind = {}
    edges, seen_edge = [], set()
    q = deque(seeds)
    print(f"시드 {len(seeds)}개, 최대 {a.hops} hop, 새 주소 최대 {a.max_nodes}개 추적 시작")
    est = (len(seeds) + a.max_nodes) * 4 / a.rps / 60
    print(f"예상 소요 시간: 최대 약 {est:.0f}분 (API 호출 한도 {a.rps}회/초 기준)")

    limit = len(seeds) + a.max_nodes   # 시드가 많아도 확장이 막히지 않도록 시드 수는 따로 센다
    while q and len(hop) < limit:
        addr = q.popleft()
        if addr not in kind:
            kind[addr] = "contract" if es.is_contract(addr) else "EOA"
        # 컨트랙트(거래소 핫월렛 제외 대부분 DEX/브릿지)는 확장하지 않고 종착점으로 둔다
        if kind[addr] == "contract" and hop[addr] > 0:
            continue
        if hop[addr] >= a.hops:
            continue
        if addr in labels and hop[addr] > 0 and labels[addr] != "FBI seed":
            continue  # 라벨 붙은 거래소/브릿지 = 종착점
        print(f"[hop {hop[addr]}] {addr}  (노드 {len(hop)}, API {es.calls})")
        outs = outgoing(es, addr, a.since_block, a.min_eth)
        # 수신자별 합산 후 큰 순서로 상위만 확장
        by_dst = defaultdict(float)
        for e in outs:
            by_dst[e["dst"]] += eth_equiv(e)
        keep = set(sorted(by_dst, key=by_dst.get, reverse=True)[: a.max_children])
        for e in outs:
            key = (e["tx"], e["dst"], e["asset"])
            if key in seen_edge:
                continue
            seen_edge.add(key)
            edges.append(e)
            d = e["dst"]
            if d in keep and d not in hop:
                hop[d] = hop[addr] + 1
                q.append(d)

    # 종착 노드 타입 확인
    for addr in list(hop):
        if addr not in kind:
            kind[addr] = "contract" if es.is_contract(addr) else "EOA"

    # 집계
    tin, tout = defaultdict(float), defaultdict(float)
    fns = defaultdict(set)
    for e in edges:
        tout[e["src"]] += eth_equiv(e)
        tin[e["dst"]] += eth_equiv(e)
        if e["fn"]:
            fns[e["dst"]].add(e["fn"])

    def flag(addr):
        fl = []
        if kind.get(addr) == "contract":
            fl.append("contract")
            if any(h in f.lower() for f in fns[addr] for h in EXIT_HINTS):
                fl.append("EXIT_CANDIDATE")
        if addr in labels:
            fl.append("labeled")
        return ";".join(fl)

    os.makedirs(a.out, exist_ok=True)
    with open(os.path.join(a.out, "nodes.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["address", "hop", "type", "label", "flags",
                    "in_eth_equiv", "out_eth_equiv", "called_functions", "etherscan"])
        for addr in sorted(hop, key=lambda x: (hop[x], -tin[x])):
            w.writerow([addr, hop[addr], kind.get(addr, ""), labels.get(addr, ""),
                        flag(addr), round(tin[addr], 4), round(tout[addr], 4),
                        "|".join(sorted(fns[addr])),
                        f"https://etherscan.io/address/{addr}"])
    with open(os.path.join(a.out, "edges.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["from", "to", "asset", "amount", "time_utc", "function", "tx", "etherscan"])
        for e in edges:
            w.writerow([e["src"], e["dst"], e["asset"], e["amount"], e["time"], e["fn"],
                        e["tx"], f"https://etherscan.io/tx/{e['tx']}"])
    exits = [x for x in hop if kind.get(x) == "contract" and tin[x] > 0]
    exits.sort(key=lambda x: -tin[x])
    with open(os.path.join(a.out, "exits.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["contract", "label", "in_eth_equiv", "functions", "flags", "etherscan"])
        for x in exits:
            w.writerow([x, labels.get(x, ""), round(tin[x], 4), "|".join(sorted(fns[x])),
                        flag(x), f"https://etherscan.io/address/{x}"])

    write_graph(os.path.join(a.out, "graph.html"), hop, kind, labels, edges, tin, flag)
    print(f"\n완료: 노드 {len(hop)}, 엣지 {len(edges)}, 출구 후보 {len(exits)}, API 호출 {es.calls}")
    print(f"결과: {os.path.abspath(a.out)}/")
    if exits:
        print("\n유입 상위 컨트랙트 (출구 후보):")
        for x in exits[:10]:
            print(f"  {x}  ~{tin[x]:,.1f} ETH-eq  {labels.get(x, '')}  {flag(x)}")


def write_graph(path, hop, kind, labels, edges, tin, flag):
    colors = ["#d62728", "#ff7f0e", "#e6b800", "#2ca02c", "#1f77b4", "#9467bd"]
    nodes = []
    for a_, h in hop.items():
        fl = flag(a_)
        nodes.append({
            "id": a_,
            "label": (labels.get(a_) or a_[:8] + "…") + f"\nhop {h}",
            "title": f"{a_}\n{kind.get(a_, '')} {fl}\n유입 ~{tin[a_]:.2f} ETH-eq",
            "color": "#111" if "EXIT_CANDIDATE" in fl else colors[min(h, len(colors) - 1)],
            "shape": "box" if kind.get(a_) == "contract" else "dot",
            "value": max(1.0, tin[a_]),
        })
    agg = defaultdict(lambda: [0.0, set()])
    for e in edges:
        if e["src"] in hop and e["dst"] in hop:
            k = (e["src"], e["dst"])
            agg[k][0] += eth_equiv(e)
            agg[k][1].add(e["asset"])
    links = [{"from": s, "to": d, "arrows": "to", "value": v,
              "title": f"~{v:,.2f} ETH-eq ({', '.join(sorted(assets))})"}
             for (s, d), (v, assets) in agg.items()]
    html = f"""<!doctype html><html lang="ko"><head><meta charset="utf-8">
<title>ETH 자금 흐름</title>
<script src="https://unpkg.com/vis-network@9.1.9/standalone/umd/vis-network.min.js"></script>
<style>body{{margin:0;font-family:sans-serif}}#g{{width:100%;height:720px}}
#h{{padding:8px 12px;font-size:13px;background:#f4f4f4}}</style></head><body>
<div id="h">빨강=시드(hop0) → 주황/노랑/초록=다음 hop · 상자=컨트랙트 · 검정 상자=출구 후보 · 노드 더블클릭 시 Etherscan</div>
<div id="g"></div><script>
const nodes=new vis.DataSet({json.dumps(nodes)});
const edges=new vis.DataSet({json.dumps(links)});
const net=new vis.Network(document.getElementById('g'),{{nodes,edges}},{{
 layout:{{hierarchical:{{direction:'LR',sortMethod:'directed',levelSeparation:260}}}},
 physics:false,nodes:{{font:{{size:11}},scaling:{{min:6,max:40}}}},
 edges:{{scaling:{{min:1,max:12}},smooth:{{type:'cubicBezier'}}}}}});
net.on('doubleClick',p=>{{if(p.nodes.length)window.open('https://etherscan.io/address/'+p.nodes[0])}});
</script></body></html>"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile trace_flows.py
#!/usr/bin/env python3
"""
trace_flows.py - 공개된 시드 주소(예: FBI 공개 목록)에서 출발해
이더리움 자금 흐름을 N-hop까지 추적하고, 출구(브릿지/스왑/거래소 등
컨트랙트)로 보이는 지점을 표시한다.

데이터 출처: Etherscan API V2 (무료 키: https://etherscan.io/myapikey)

사용법:
  export ETHERSCAN_API_KEY=키
  python3 trace_flows.py addresses.txt --since-block 23690000 --hops 3

  Colab: 왼쪽 열쇠(🔑) 아이콘 > 보안 비밀에 ETHERSCAN_API_KEY 를 등록하면
         키를 따로 넘기지 않아도 자동으로 읽는다.
         !python trace_flows.py addresses.txt --hops 2

입력:
  addresses.txt   한 줄에 주소 하나 (# 주석 가능)
  --labels FILE   (선택) address,label CSV. 알려진 거래소/브릿지 주소 등

출력 (out/ 폴더):
  nodes.csv       주소, hop, EOA/컨트랙트, 라벨, 유입/유출 합계
  edges.csv       송금 내역 (from, to, 자산, 수량, tx, 시각, 호출 함수)
  exits.csv       자금이 흘러든 컨트랙트(=출구 후보) 유입액 순위
  graph.html      브라우저로 여는 인터랙티브 흐름 그래프

주의: 공개 블록체인 데이터 조회만 수행한다. 이 주소들로 송금하거나
상호작용하지 말 것. 결과는 단서일 뿐이며, 의미 있는 발견은
tips.fbi.gov / ic3.gov 에 근거(tx 해시)와 함께 제보할 것.
"""
import argparse
import csv
import json
import os
import sys
import time
import urllib.parse
import urllib.request
from collections import defaultdict, deque
from datetime import datetime, timezone

API = "https://api.etherscan.io/v2/api"
CHAIN_ID = 1  # Ethereum mainnet

# 추적할 토큰과 최소 수량 (이보다 작은 이동은 무시)
TRACK_TOKENS = {
    "WETH": 1.0, "stETH": 1.0, "wstETH": 1.0, "cbETH": 1.0, "rETH": 1.0,
    "USDT": 10_000, "USDC": 10_000, "DAI": 10_000,
    "WBTC": 0.05, "cbBTC": 0.05,
}

# 호출 함수명에 이런 단어가 있으면 브릿지/스왑/믹서 출구 후보로 표시
EXIT_HINTS = [
    "bridge", "swap", "transferout", "outbound", "deposit", "relay",
    "send", "lock", "exactinput", "multicall", "execute", "fill",
]


class Etherscan:
    def __init__(self, key, rps=4.0):
        self.key = key
        self.min_gap = 1.0 / rps
        self.last = 0.0
        self.calls = 0

    def get(self, **params):
        params.update(chainid=CHAIN_ID, apikey=self.key)
        url = API + "?" + urllib.parse.urlencode(params)
        for attempt in range(6):
            wait = self.min_gap - (time.time() - self.last)
            if wait > 0:
                time.sleep(wait)
            self.last = time.time()
            self.calls += 1
            try:
                with urllib.request.urlopen(url, timeout=30) as r:
                    data = json.loads(r.read().decode())
            except Exception as e:
                print(f"  ! 요청 오류({e}), 재시도", file=sys.stderr)
                time.sleep(2 * (attempt + 1))
                continue
            res = data.get("result")
            if isinstance(res, str) and "rate limit" in res.lower():
                time.sleep(1.5 * (attempt + 1))
                continue
            if data.get("status") == "0":
                msg = str(data.get("message", "")) + str(res)
                if "No transactions found" in msg or "No records found" in msg:
                    return []
                if "Invalid API Key" in msg or "Missing" in msg:
                    sys.exit(f"API 키 오류: {msg}")
                print(f"  ! API 응답: {msg[:120]}", file=sys.stderr)
                return []
            return res
        print("  ! 재시도 한도 초과, 건너뜀", file=sys.stderr)
        return []

    def txlist(self, addr, start):
        return self.get(module="account", action="txlist", address=addr,
                        startblock=start, endblock=99999999,
                        page=1, offset=1000, sort="asc")

    def internal(self, addr, start):
        return self.get(module="account", action="txlistinternal", address=addr,
                        startblock=start, endblock=99999999,
                        page=1, offset=1000, sort="asc")

    def tokentx(self, addr, start):
        return self.get(module="account", action="tokentx", address=addr,
                        startblock=start, endblock=99999999,
                        page=1, offset=1000, sort="asc")

    def is_contract(self, addr):
        res = self.get(module="proxy", action="eth_getCode",
                       address=addr, tag="latest")
        return isinstance(res, str) and res not in ("0x", "0x0", "")


def get_api_key(cli_key=None):
    """--api-key > 환경변수 > Colab 보안 비밀 순으로 API 키를 찾는다."""
    if cli_key:
        return cli_key
    if os.environ.get("ETHERSCAN_API_KEY"):
        return os.environ["ETHERSCAN_API_KEY"]
    try:
        from google.colab import userdata  # Colab에서만 존재
        return userdata.get("ETHERSCAN_API_KEY")
    except Exception:
        return None


def ts(t):
    try:
        return datetime.fromtimestamp(int(t), tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return ""


def fn_name(tx):
    f = (tx.get("functionName") or "").split("(")[0]
    return f


def outgoing(es, addr, start, min_eth):
    """addr에서 나간 의미 있는 이동 목록"""
    edges = []
    for tx in es.txlist(addr, start):
        if tx.get("from", "").lower() != addr or tx.get("isError") == "1":
            continue
        to = (tx.get("to") or "").lower()
        if not to:
            continue
        val = int(tx.get("value", "0")) / 1e18
        f = fn_name(tx)
        # ETH 이동이거나, 값이 없어도 컨트랙트 호출(토큰 승인/브릿지 호출) 기록
        if val >= min_eth or (val > 0 and f):
            edges.append(dict(src=addr, dst=to, asset="ETH", amount=val,
                              tx=tx["hash"], time=ts(tx.get("timeStamp")), fn=f))
    for tx in es.internal(addr, start):
        if tx.get("from", "").lower() != addr or tx.get("isError") == "1":
            continue
        to = (tx.get("to") or "").lower()
        val = int(tx.get("value", "0")) / 1e18
        if to and val >= min_eth:
            edges.append(dict(src=addr, dst=to, asset="ETH(internal)", amount=val,
                              tx=tx["hash"], time=ts(tx.get("timeStamp")), fn=""))
    for tx in es.tokentx(addr, start):
        if tx.get("from", "").lower() != addr:
            continue
        sym = tx.get("tokenSymbol", "")
        if sym not in TRACK_TOKENS:
            continue
        dec = int(tx.get("tokenDecimal") or 18)
        amt = int(tx.get("value", "0")) / (10 ** dec)
        if amt >= TRACK_TOKENS[sym]:
            edges.append(dict(src=addr, dst=tx["to"].lower(), asset=sym, amount=amt,
                              tx=tx["hash"], time=ts(tx.get("timeStamp")), fn=""))
    return edges


def eth_equiv(e):
    """팬아웃 정렬용 대략적 크기 (ETH 계열만 비교, 나머지는 보조)"""
    if e["asset"].startswith("ETH") or e["asset"] in ("WETH", "stETH", "wstETH", "cbETH", "rETH"):
        return e["amount"]
    if e["asset"] in ("WBTC", "cbBTC"):
        return e["amount"] * 30
    return e["amount"] / 3000  # 스테이블코인 대략 환산


def load_list(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.split("#")[0].strip().strip(",")
            if line.lower().startswith("0x") and len(line) == 42:
                out.append(line.lower())
    return list(dict.fromkeys(out))


def load_labels(path):
    labels = {}
    if not path:
        return labels
    with open(path, encoding="utf-8") as f:
        for row in csv.reader(f):
            if len(row) >= 2 and row[0].lower().startswith("0x"):
                labels[row[0].strip().lower()] = row[1].strip()
    return labels


def main(argv=None):
    ap = argparse.ArgumentParser(description="공개 시드 주소 기반 ETH 자금 흐름 추적")
    ap.add_argument("addresses", help="시드 주소 파일 (한 줄에 하나)")
    ap.add_argument("--api-key", default=None, help="생략 시 환경변수/Colab 보안 비밀 사용")
    ap.add_argument("--since-block", type=int, default=0,
                    help="이 블록 이후 거래만 조회 (해킹 발생 블록 권장)")
    ap.add_argument("--hops", type=int, default=3, help="추적 깊이")
    ap.add_argument("--min-eth", type=float, default=1.0, help="최소 ETH 이동량")
    ap.add_argument("--max-children", type=int, default=15,
                    help="노드당 따라갈 최대 수신 주소 수 (큰 금액 순)")
    ap.add_argument("--max-nodes", type=int, default=600,
                    help="시드 외에 새로 추적할 최대 주소 수")
    ap.add_argument("--labels", help="address,label CSV (선택)")
    ap.add_argument("--rps", type=float, default=4.0, help="초당 API 호출 (무료 5)")
    ap.add_argument("--out", default="out")
    a = ap.parse_args(argv)

    a.api_key = get_api_key(a.api_key)
    if not a.api_key:
        sys.exit("API 키가 없습니다. --api-key, 환경변수 ETHERSCAN_API_KEY, "
                 "또는 Colab 보안 비밀(ETHERSCAN_API_KEY) 중 하나로 넘겨 주세요.")

    es = Etherscan(a.api_key, a.rps)
    seeds = load_list(a.addresses)
    if not seeds:
        sys.exit("시드 주소를 찾지 못했습니다.")
    labels = load_labels(a.labels)
    for s in seeds:
        labels.setdefault(s, "FBI seed")

    hop = {s: 0 for s in seeds}
    kind = {}
    edges, seen_edge = [], set()
    q = deque(seeds)
    print(f"시드 {len(seeds)}개, 최대 {a.hops} hop, 새 주소 최대 {a.max_nodes}개 추적 시작")
    est = (len(seeds) + a.max_nodes) * 4 / a.rps / 60
    print(f"예상 소요 시간: 최대 약 {est:.0f}분 (API 호출 한도 {a.rps}회/초 기준)")

    limit = len(seeds) + a.max_nodes   # 시드가 많아도 확장이 막히지 않도록 시드 수는 따로 센다
    while q and len(hop) < limit:
        addr = q.popleft()
        if addr not in kind:
            kind[addr] = "contract" if es.is_contract(addr) else "EOA"
        # 컨트랙트(거래소 핫월렛 제외 대부분 DEX/브릿지)는 확장하지 않고 종착점으로 둔다
        if kind[addr] == "contract" and hop[addr] > 0:
            continue
        if hop[addr] >= a.hops:
            continue
        if addr in labels and hop[addr] > 0 and labels[addr] != "FBI seed":
            continue  # 라벨 붙은 거래소/브릿지 = 종착점
        print(f"[hop {hop[addr]}] {addr}  (노드 {len(hop)}, API {es.calls})")
        outs = outgoing(es, addr, a.since_block, a.min_eth)
        # 수신자별 합산 후 큰 순서로 상위만 확장
        by_dst = defaultdict(float)
        for e in outs:
            by_dst[e["dst"]] += eth_equiv(e)
        keep = set(sorted(by_dst, key=by_dst.get, reverse=True)[: a.max_children])
        for e in outs:
            key = (e["tx"], e["dst"], e["asset"])
            if key in seen_edge:
                continue
            seen_edge.add(key)
            edges.append(e)
            d = e["dst"]
            if d in keep and d not in hop:
                hop[d] = hop[addr] + 1
                q.append(d)

    # 종착 노드 타입 확인
    for addr in list(hop):
        if addr not in kind:
            kind[addr] = "contract" if es.is_contract(addr) else "EOA"

    # 집계
    tin, tout = defaultdict(float), defaultdict(float)
    fns = defaultdict(set)
    for e in edges:
        tout[e["src"]] += eth_equiv(e)
        tin[e["dst"]] += eth_equiv(e)
        if e["fn"]:
            fns[e["dst"]].add(e["fn"])

    def flag(addr):
        fl = []
        if kind.get(addr) == "contract":
            fl.append("contract")
            if any(h in f.lower() for f in fns[addr] for h in EXIT_HINTS):
                fl.append("EXIT_CANDIDATE")
        if addr in labels:
            fl.append("labeled")
        return ";".join(fl)

    os.makedirs(a.out, exist_ok=True)
    with open(os.path.join(a.out, "nodes.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["address", "hop", "type", "label", "flags",
                    "in_eth_equiv", "out_eth_equiv", "called_functions", "etherscan"])
        for addr in sorted(hop, key=lambda x: (hop[x], -tin[x])):
            w.writerow([addr, hop[addr], kind.get(addr, ""), labels.get(addr, ""),
                        flag(addr), round(tin[addr], 4), round(tout[addr], 4),
                        "|".join(sorted(fns[addr])),
                        f"https://etherscan.io/address/{addr}"])
    with open(os.path.join(a.out, "edges.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["from", "to", "asset", "amount", "time_utc", "function", "tx", "etherscan"])
        for e in edges:
            w.writerow([e["src"], e["dst"], e["asset"], e["amount"], e["time"], e["fn"],
                        e["tx"], f"https://etherscan.io/tx/{e['tx']}"])
    exits = [x for x in hop if kind.get(x) == "contract" and tin[x] > 0]
    exits.sort(key=lambda x: -tin[x])
    with open(os.path.join(a.out, "exits.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["contract", "label", "in_eth_equiv", "functions", "flags", "etherscan"])
        for x in exits:
            w.writerow([x, labels.get(x, ""), round(tin[x], 4), "|".join(sorted(fns[x])),
                        flag(x), f"https://etherscan.io/address/{x}"])

    write_graph(os.path.join(a.out, "graph.html"), hop, kind, labels, edges, tin, flag)
    print(f"\n완료: 노드 {len(hop)}, 엣지 {len(edges)}, 출구 후보 {len(exits)}, API 호출 {es.calls}")
    print(f"결과: {os.path.abspath(a.out)}/")
    if exits:
        print("\n유입 상위 컨트랙트 (출구 후보):")
        for x in exits[:10]:
            print(f"  {x}  ~{tin[x]:,.1f} ETH-eq  {labels.get(x, '')}  {flag(x)}")


def write_graph(path, hop, kind, labels, edges, tin, flag):
    colors = ["#d62728", "#ff7f0e", "#e6b800", "#2ca02c", "#1f77b4", "#9467bd"]
    nodes = []
    for a_, h in hop.items():
        fl = flag(a_)
        nodes.append({
            "id": a_,
            "label": (labels.get(a_) or a_[:8] + "…") + f"\nhop {h}",
            "title": f"{a_}\n{kind.get(a_, '')} {fl}\n유입 ~{tin[a_]:.2f} ETH-eq",
            "color": "#111" if "EXIT_CANDIDATE" in fl else colors[min(h, len(colors) - 1)],
            "shape": "box" if kind.get(a_) == "contract" else "dot",
            "value": max(1.0, tin[a_]),
        })
    agg = defaultdict(lambda: [0.0, set()])
    for e in edges:
        if e["src"] in hop and e["dst"] in hop:
            k = (e["src"], e["dst"])
            agg[k][0] += eth_equiv(e)
            agg[k][1].add(e["asset"])
    links = [{"from": s, "to": d, "arrows": "to", "value": v,
              "title": f"~{v:,.2f} ETH-eq ({', '.join(sorted(assets))})"}
             for (s, d), (v, assets) in agg.items()]
    html = f"""<!doctype html><html lang="ko"><head><meta charset="utf-8">
<title>ETH 자금 흐름</title>
<script src="https://unpkg.com/vis-network@9.1.9/standalone/umd/vis-network.min.js"></script>
<style>body{{margin:0;font-family:sans-serif}}#g{{width:100%;height:720px}}
#h{{padding:8px 12px;font-size:13px;background:#f4f4f4}}</style></head><body>
<div id="h">빨강=시드(hop0) → 주황/노랑/초록=다음 hop · 상자=컨트랙트 · 검정 상자=출구 후보 · 노드 더블클릭 시 Etherscan</div>
<div id="g"></div><script>
const nodes=new vis.DataSet({json.dumps(nodes)});
const edges=new vis.DataSet({json.dumps(links)});
const net=new vis.Network(document.getElementById('g'),{{nodes,edges}},{{
 layout:{{hierarchical:{{direction:'LR',sortMethod:'directed',levelSeparation:260}}}},
 physics:false,nodes:{{font:{{size:11}},scaling:{{min:6,max:40}}}},
 edges:{{scaling:{{min:1,max:12}},smooth:{{type:'cubicBezier'}}}}}});
net.on('doubleClick',p=>{{if(p.nodes.length)window.open('https://etherscan.io/address/'+p.nodes[0])}});
</script></body></html>"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile analyze.py
#!/usr/bin/env python3
"""
analyze.py - trace_flows.py(또는 make_sample.py) 결과 폴더를 읽어
세탁 패턴 지표를 계산하고, 여러 사례를 나란히 비교하는 리포트를 만든다.

사용법:
  python3 analyze.py 이름=결과폴더 [이름=결과폴더 ...] [--out report]
  예) python3 analyze.py Bybit=out_bybit Binance=out_binance

출력:
  report/metrics.csv   사례별 지표 표
  report/report.html   비교 리포트 (차트 + 표, 브라우저로 열기)

지표 정의
  추적 유출액        시드(hop0)에서 나간 총액 (ETH 환산)
  출구 도달 비율     컨트랙트(브릿지/스왑 등)로 들어간 총액 / 추적 유출액
  첫 출구까지        첫 시드 유출 ~ 첫 컨트랙트 유입 (시간)
  T50 / T90          최종 출구 유입액의 50% / 90%가 도달하기까지 (시간)
  평균 팬아웃        유출이 있는 EOA 하나가 돈을 보낸 서로 다른 EOA 수 평균
  경유 지갑 비율     유출의 80% 이상을 다른 개인 지갑 한 곳으로 넘긴 EOA 비율 (필 체인/경유 지갑)
  최장 경유 체인     경유 지갑이 연속으로 이어진 최대 길이
  중앙 보유 시간     EOA가 처음 받은 뒤 처음 보내기까지 걸린 시간의 중앙값 (분)
  자산 구성          ETH / 스테이블코인 / BTC 래핑 / 기타 비중
"""
import argparse
import csv
import json
import os
import statistics
import sys
from collections import defaultdict
from datetime import datetime, timezone

ETH_LIKE = {"ETH", "ETH(internal)", "WETH", "stETH", "wstETH", "cbETH", "rETH"}
BTC_LIKE = {"WBTC", "cbBTC"}
STABLE = {"USDT", "USDC", "DAI"}


def eth_equiv(asset, amount):
    if asset in ETH_LIKE:
        return amount
    if asset in BTC_LIKE:
        return amount * 30
    return amount / 3000


def asset_group(asset):
    if asset in ETH_LIKE:
        return "ETH"
    if asset in STABLE:
        return "스테이블코인"
    if asset in BTC_LIKE:
        return "BTC 래핑"
    return "기타"


def parse_t(s):
    try:
        return datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc).timestamp()
    except Exception:
        return None


def load(folder):
    nodes, edges = {}, []
    with open(os.path.join(folder, "nodes.csv"), encoding="utf-8") as f:
        for r in csv.DictReader(f):
            nodes[r["address"]] = dict(hop=int(r["hop"]), type=r["type"],
                                       flags=r.get("flags", ""), label=r.get("label", ""))
    with open(os.path.join(folder, "edges.csv"), encoding="utf-8") as f:
        for r in csv.DictReader(f):
            t = parse_t(r["time_utc"])
            if t is None:
                continue
            amt = float(r["amount"])
            edges.append(dict(src=r["from"], dst=r["to"], asset=r["asset"], amount=amt,
                              v=eth_equiv(r["asset"], amt), t=t, fn=r.get("function", "")))
    edges.sort(key=lambda e: e["t"])
    return nodes, edges


def is_synthetic(nodes):
    """make_sample.py가 만든 데이터는 라벨에 '(가상)'이 붙어 있다."""
    return any("(가상)" in (n.get("label") or "") for n in nodes.values())


def analyze(nodes, edges):
    seeds = {a for a, n in nodes.items() if n["hop"] == 0}
    is_contract = lambda a: nodes.get(a, {}).get("type") == "contract"
    is_eoa = lambda a: nodes.get(a, {}).get("type") == "EOA"

    seed_edges = [e for e in edges if e["src"] in seeds]
    if not seed_edges:
        return None
    t0 = seed_edges[0]["t"]
    seed_out = sum(e["v"] for e in seed_edges)

    exit_edges = [e for e in edges if is_contract(e["dst"])]
    exit_total = sum(e["v"] for e in exit_edges)

    # 누적 출구 도달 곡선 (시간 단위, 추적 유출액 대비 비율)
    curve, cum = [[0.0, 0.0]], 0.0
    for e in exit_edges:
        cum += e["v"]
        curve.append([round((e["t"] - t0) / 3600, 3), round(min(cum / seed_out, 1.0), 4)])

    def t_reach(frac):
        target, c = exit_total * frac, 0.0
        for e in exit_edges:
            c += e["v"]
            if c >= target:
                return (e["t"] - t0) / 3600
        return None

    # 노드 수 (hop별)
    max_hop = max(n["hop"] for n in nodes.values())
    per_hop = []
    for h in range(max_hop + 1):
        per_hop.append(dict(
            hop=h,
            eoa=sum(1 for n in nodes.values() if n["hop"] == h and n["type"] == "EOA"),
            contract=sum(1 for n in nodes.values() if n["hop"] == h and n["type"] == "contract"),
        ))

    # 유입/유출 집계
    out_by = defaultdict(lambda: defaultdict(float))
    in_total, out_total = defaultdict(float), defaultdict(float)
    first_in, first_out = {}, {}
    for e in edges:
        out_by[e["src"]][e["dst"]] += e["v"]
        out_total[e["src"]] += e["v"]
        in_total[e["dst"]] += e["v"]
        first_in.setdefault(e["dst"], e["t"])
        first_out.setdefault(e["src"], e["t"])

    fanouts = [sum(1 for d in out_by[a] if is_eoa(d))
               for a in nodes if is_eoa(a) and out_total[a] > 0]
    fanouts = [x for x in fanouts if x > 0]

    # 경유 지갑: hop>=1 EOA, 유출의 80% 이상이 한 곳으로
    passthru = {}
    for a, n in nodes.items():
        if n["hop"] >= 1 and n["type"] == "EOA" and out_total[a] > 0:
            dst, v = max(out_by[a].items(), key=lambda kv: kv[1])
            if v >= 0.8 * out_total[a] and is_eoa(dst):
                passthru[a] = dst
    candidates = [a for a, n in nodes.items() if n["hop"] >= 1 and n["type"] == "EOA" and out_total[a] > 0]

    def chain_len(a, seen):
        if a not in passthru or a in seen:
            return 0
        seen.add(a)
        return 1 + chain_len(passthru[a], seen)
    longest = max((chain_len(a, set()) for a in passthru), default=0)

    holds = [(first_out[a] - first_in[a]) / 60 for a in nodes
             if is_eoa(a) and nodes[a]["hop"] >= 1 and a in first_in and a in first_out
             and first_out[a] >= first_in[a]]

    mix = defaultdict(float)
    for e in edges:
        mix[asset_group(e["asset"])] += e["v"]
    mix_total = sum(mix.values()) or 1

    fn_val = defaultdict(float)
    for e in exit_edges:
        fn_val[e["fn"] or "(직접 송금)"] += e["v"]

    return dict(
        seed_count=len(seeds),
        node_count=len(nodes),
        edge_count=len(edges),
        seed_out=seed_out,
        exit_share=exit_total / seed_out if seed_out else 0,
        first_exit_h=(exit_edges[0]["t"] - t0) / 3600 if exit_edges else None,
        t50_h=t_reach(0.5) if exit_edges else None,
        t90_h=t_reach(0.9) if exit_edges else None,
        mean_fanout=statistics.mean(fanouts) if fanouts else 0,
        passthru_ratio=len(passthru) / len(candidates) if candidates else 0,
        longest_chain=longest,
        median_hold_min=statistics.median(holds) if holds else None,
        mix={k: v / mix_total for k, v in sorted(mix.items(), key=lambda kv: -kv[1])},
        top_exit_fns=sorted(fn_val.items(), key=lambda kv: -kv[1])[:5],
        per_hop=per_hop,
        curve=curve,
        start_utc=datetime.fromtimestamp(t0, tz=timezone.utc).strftime("%Y-%m-%d %H:%M"),
    )


ROWS = [
    ("seed_count", "시드 주소 수", "{:,.0f}"),
    ("node_count", "추적된 주소 수", "{:,.0f}"),
    ("edge_count", "송금 건수", "{:,.0f}"),
    ("seed_out", "추적 유출액 (ETH 환산)", "{:,.1f}"),
    ("exit_share", "출구 도달 비율", "{:.1%}"),
    ("first_exit_h", "첫 출구까지 (시간)", "{:.2f}"),
    ("t50_h", "T50 (시간)", "{:.2f}"),
    ("t90_h", "T90 (시간)", "{:.2f}"),
    ("mean_fanout", "평균 팬아웃", "{:.2f}"),
    ("passthru_ratio", "경유 지갑 비율", "{:.1%}"),
    ("longest_chain", "최장 경유 체인", "{:,.0f}"),
    ("median_hold_min", "중앙 보유 시간 (분)", "{:.1f}"),
]


def fmt(v, f):
    return "—" if v is None else f.format(v)


def main(argv=None):
    ap = argparse.ArgumentParser(description="세탁 패턴 지표 계산 및 사례 비교 리포트")
    ap.add_argument("cases", nargs="+", help="이름=결과폴더")
    ap.add_argument("--out", default="report")
    a = ap.parse_args(argv)

    results = []
    for c in a.cases:
        if "=" not in c:
            sys.exit(f"'이름=폴더' 형식이어야 합니다: {c}")
        name, folder = c.split("=", 1)
        if not os.path.exists(os.path.join(folder, "nodes.csv")):
            sys.exit(f"{folder}/nodes.csv 가 없습니다. make_sample.py 또는 trace_flows.py를 먼저 실행하세요.")
        nodes, edges = load(folder)
        m = analyze(nodes, edges)
        if m is None:
            print(f"! {name}: 시드 유출 기록이 없어 건너뜀", file=sys.stderr)
            continue
        m["name"] = name
        m["synthetic"] = is_synthetic(nodes)
        results.append(m)
    if not results:
        sys.exit("분석할 사례가 없습니다.")

    os.makedirs(a.out, exist_ok=True)
    with open(os.path.join(a.out, "metrics.csv"), "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["지표"] + [r["name"] for r in results])
        for key, label, f_ in ROWS:
            w.writerow([label] + [fmt(r[key], f_) for r in results])
        groups = sorted({g for r in results for g in r["mix"]})
        for g in groups:
            w.writerow([f"자산 비중: {g}"] + [f"{r['mix'].get(g, 0):.1%}" for r in results])

    # 콘솔 요약
    print("\n" + " | ".join(["지표".ljust(22)] + [r["name"][:14].ljust(14) for r in results]))
    for key, label, f_ in ROWS:
        print(" | ".join([label.ljust(22)] + [fmt(r[key], f_).ljust(14) for r in results]))

    write_report(os.path.join(a.out, "report.html"), results)
    write_tistory(os.path.join(a.out, "report_tistory.html"), results)
    print(f"\n리포트: {os.path.abspath(os.path.join(a.out, 'report.html'))}")
    print(f"블로그용: {os.path.abspath(os.path.join(a.out, 'report_tistory.html'))}"
          "  (티스토리 HTML 모드에 붙여넣기)")


def write_report(path, results):
    table_rows = []
    for key, label, f_ in ROWS:
        table_rows.append([label] + [fmt(r[key], f_) for r in results])
    groups = sorted({g for r in results for g in r["mix"]})
    for g in groups:
        table_rows.append([f"자산 비중: {g}"] + [f"{r['mix'].get(g, 0):.1%}" for r in results])
    data = dict(
        cases=[dict(name=r["name"], curve=r["curve"], per_hop=r["per_hop"],
                    start=r["start_utc"], fns=r["top_exit_fns"]) for r in results],
        head=["지표"] + [r["name"] for r in results],
        rows=table_rows,
    )
    with open(path, "w", encoding="utf-8") as f:
        f.write(REPORT_HTML.replace("__DATA__", json.dumps(data, ensure_ascii=False)))


# ---------------------------------------------------------------------------
# 블로그(티스토리)용 리포트
#   - <html>/<head>/<body> 없이 본문에 바로 붙이는 HTML 조각
#   - 스크립트 없이 정적 SVG로 차트를 그려, 블로그가 스크립트를 막아도 보인다
#   - 스타일을 모두 인라인으로 넣어 블로그 테마 CSS와 섞이지 않는다
# ---------------------------------------------------------------------------
T_COL = ["#2a78d6", "#eb6834", "#1baf7a"]
T_INK, T_INK2, T_MUTED, T_GRID, T_AXIS = "#0b0b0b", "#52514e", "#8a8984", "#e4e3df", "#b9b8b2"
T_FONT = ("-apple-system,BlinkMacSystemFont,'Apple SD Gothic Neo','Malgun Gothic',"
          "'Noto Sans KR',sans-serif")


def _esc(x):
    return (str(x).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
            .replace('"', "&quot;"))


def _nice_max(v):
    import math
    if v <= 0:
        return 1
    p = 10 ** math.floor(math.log10(v))
    for m in (1, 2, 2.5, 5, 10):
        if m * p >= v:
            return m * p
    return 10 * p


def _num(v):
    return f"{v:g}" if v < 10 else f"{v:,.0f}"


def _legend(results):
    items = "".join(
        f'<span style="margin-right:16px;white-space:nowrap">'
        f'<span style="display:inline-block;width:12px;height:12px;border-radius:3px;'
        f'background:{T_COL[i % 3]};margin-right:6px;vertical-align:-1px"></span>{_esc(r["name"])}</span>'
        for i, r in enumerate(results))
    return f'<p style="margin:6px 0 4px;font-size:13px;color:{T_INK2}">{items}</p>'


def _svg_curve(results):
    W, H = 520, 280
    l, r_, t, b = 42, 62, 14, 42
    xmax = _nice_max(max(r["curve"][-1][0] for r in results) * 1.02)
    X = lambda v: l + v / xmax * (W - l - r_)
    Y = lambda v: H - b - v * (H - t - b)
    o = [f'<svg viewBox="0 0 {W} {H}" width="100%" role="img" aria-label="누적 출구 도달 비율" '
         f'style="display:block;max-width:100%;height:auto;font-family:{T_FONT}">']
    for i in range(5):
        v = i / 4
        o.append(f'<line x1="{l}" x2="{W - r_}" y1="{Y(v):.1f}" y2="{Y(v):.1f}" stroke="{T_GRID}"/>')
        o.append(f'<text x="{l - 8}" y="{Y(v) + 4:.1f}" text-anchor="end" font-size="12.5" '
                 f'fill="{T_INK2}">{int(v * 100)}%</text>')
    for i in range(6):
        v = xmax * i / 5
        o.append(f'<text x="{X(v):.1f}" y="{H - b + 18}" text-anchor="middle" font-size="12.5" '
                 f'fill="{T_INK2}">{_num(v)}h</text>')
    o.append(f'<line x1="{l}" x2="{W - r_}" y1="{Y(0):.1f}" y2="{Y(0):.1f}" stroke="{T_AXIS}"/>')
    o.append(f'<text x="{W - r_}" y="{H - 4}" text-anchor="end" font-size="12.5" fill="{T_MUTED}">'
             f'시드 첫 유출 이후 경과 시간</text>')
    ends = []
    for i, r in enumerate(results):
        d, first = "", True
        for tt, vv in r["curve"]:
            d += f"M{X(tt):.1f},{Y(vv):.1f}" if first else f"H{X(tt):.1f}V{Y(vv):.1f}"
            first = False
        d += f"H{X(xmax):.1f}"
        o.append(f'<path d="{d}" fill="none" stroke="{T_COL[i % 3]}" stroke-width="2" '
                 f'stroke-linejoin="round"/>')
        ends.append([Y(r["curve"][-1][1]) + 4, i])
    ends.sort()
    for k in range(1, len(ends)):          # 끝 라벨 겹침 방지
        if ends[k][0] - ends[k - 1][0] < 14:
            ends[k][0] = ends[k - 1][0] + 14
    for y, i in ends:
        o.append(f'<text x="{W - r_ + 6}" y="{y:.1f}" font-size="13" fill="{T_INK}">'
                 f'{_esc(results[i]["name"])}</text>')
    o.append("</svg>")
    return "".join(o)


def _svg_bars(results):
    W, H = 520, 250
    l, r_, t, b = 38, 6, 16, 42
    hops = max(len(r["per_hop"]) for r in results)
    ymax = _nice_max(max(h["eoa"] for r in results for h in r["per_hop"]))
    Y = lambda v: H - b - v / ymax * (H - t - b)
    n = len(results)
    gw = (W - l - r_) / hops
    bw = min(34, (gw * 0.72 - 2 * (n - 1)) / n)
    step = max(1, -(-34 // int(gw))) if gw < 34 else 1
    o = [f'<svg viewBox="0 0 {W} {H}" width="100%" role="img" aria-label="hop별 개인 지갑 수" '
         f'style="display:block;max-width:100%;height:auto;font-family:{T_FONT}">']
    yt = 4 if ymax % 4 == 0 else 5
    for i in range(yt + 1):
        v = ymax * i / yt
        o.append(f'<line x1="{l}" x2="{W - r_}" y1="{Y(v):.1f}" y2="{Y(v):.1f}" stroke="{T_GRID}"/>')
        o.append(f'<text x="{l - 8}" y="{Y(v) + 4:.1f}" text-anchor="end" font-size="12.5" '
                 f'fill="{T_INK2}">{round(v)}</text>')
    for h in range(hops):
        gx = l + gw * h + gw / 2
        x0 = gx - (n * bw + 2 * (n - 1)) / 2
        if h % step == 0:
            lab = f"hop {h}" if gw >= 44 else str(h)
            o.append(f'<text x="{gx:.1f}" y="{H - b + 18}" text-anchor="middle" font-size="12.5" '
                     f'fill="{T_INK2}">{lab}</text>')
        for i, r in enumerate(results):
            v = r["per_hop"][h]["eoa"] if h < len(r["per_hop"]) else 0
            if v <= 0:
                continue
            x, y = x0 + i * (bw + 2), Y(v)
            hgt = H - b - y
            rr = min(4, hgt, bw / 2)
            o.append(f'<path d="M{x:.1f},{H - b}V{y + rr:.1f}Q{x:.1f},{y:.1f} {x + rr:.1f},{y:.1f}'
                     f'H{x + bw - rr:.1f}Q{x + bw:.1f},{y:.1f} {x + bw:.1f},{y + rr:.1f}V{H - b}Z" '
                     f'fill="{T_COL[i % 3]}"><title>{_esc(r["name"])} · hop {h}: 지갑 {v}개</title></path>')
    o.append(f'<line x1="{l}" x2="{W - r_}" y1="{Y(0):.1f}" y2="{Y(0):.1f}" stroke="{T_AXIS}"/>')
    o.append(f'<text x="{W - r_}" y="{H - 4}" text-anchor="end" font-size="12.5" fill="{T_MUTED}">'
             f'시드에서 몇 단계 떨어졌나 (hop)</text>')
    o.append("</svg>")
    return "".join(o)


def _speed_summary(results):
    lines = []
    for r in results:
        if r["t50_h"] is None:
            lines.append(f"<b>{_esc(r['name'])}</b>: 출구(컨트랙트)로 들어간 기록이 없다.")
            continue
        lines.append(
            f"<b>{_esc(r['name'])}</b>: 첫 출구까지 {r['first_exit_h']:.1f}시간, "
            f"출구 유입액의 절반(T50)은 {r['t50_h']:.1f}시간, 90%(T90)는 {r['t90_h']:.1f}시간 만에 도달했다. "
            f"추적한 유출액 중 {r['exit_share']:.0%}가 출구를 지났다.")
    timed = [r for r in results if r["t90_h"]]
    if len(timed) >= 2:
        fast = min(timed, key=lambda r: r["t90_h"])
        slow = max(timed, key=lambda r: r["t90_h"])
        if fast is not slow and fast["t90_h"] > 0:
            lines.append(f"T90 기준으로 {_esc(fast['name'])}가 {_esc(slow['name'])}보다 "
                         f"약 {slow['t90_h'] / fast['t90_h']:.0f}배 빨리 빠져나갔다.")
    return lines


def _spread_summary(results):
    lines = []
    for r in results:
        peak = max(r["per_hop"], key=lambda h: h["eoa"])
        lines.append(
            f"<b>{_esc(r['name'])}</b>: 지갑이 가장 많은 단계는 hop {peak['hop']}({peak['eoa']}개)이고, "
            f"지갑 하나가 평균 {r['mean_fanout']:.1f}곳으로 돈을 나눴다. "
            f"한 곳으로만 넘기는 경유 지갑이 {r['passthru_ratio']:.0%}, 최장 경유 체인은 {r['longest_chain']}단계다.")
    return lines


LI = f'style="margin:0 0 4px;font-size:14px;line-height:1.7;color:{T_INK}"'

GLOSSARY = [
    ("출구 도달 비율", "추적한 유출액 중 브릿지·스왑 등 컨트랙트로 들어간 비율. 높을수록 이미 빠져나간 돈이 많다."),
    ("T50 / T90", "출구로 들어간 돈의 50% / 90%가 도달하기까지 걸린 시간. 세탁 속도를 보여 준다."),
    ("평균 팬아웃", "개인 지갑 하나가 돈을 나눠 보낸 지갑 수의 평균. 클수록 넓게 흩뿌리는 분산형이다."),
    ("경유 지갑 비율", "받은 돈의 80% 이상을 다른 지갑 한 곳으로 넘긴 지갑의 비율. 높으면 필 체인형이다."),
    ("최장 경유 체인", "경유 지갑이 줄줄이 이어진 최대 길이."),
    ("중앙 보유 시간", "지갑이 돈을 받은 뒤 다시 보내기까지 걸린 시간의 중앙값. 짧을수록 자동화 가능성이 크다."),
]


def write_tistory(path, results):
    p = f'style="margin:0 0 10px;font-size:15px;line-height:1.7;color:{T_INK}"'
    h = f'style="margin:28px 0 6px;font-size:18px;font-weight:700;color:{T_INK}"'
    sub = f'style="margin:0 0 8px;font-size:14px;line-height:1.6;color:{T_INK2}"'
    ul = f'style="margin:8px 0 0;padding-left:20px;font-size:14px;line-height:1.7;color:{T_INK}"'
    box = (f'style="margin:0 0 18px;padding:12px 14px;border-radius:8px;background:#fdf3e7;'
           f'border:1px solid #f1d3a8;font-size:14px;line-height:1.6;color:{T_INK}"')
    o = [f'<div style="max-width:720px;margin:0 auto;padding:22px 20px;background:#ffffff;'
         f'border:1px solid {T_GRID};border-radius:12px;font-family:{T_FONT};color:{T_INK};'
         f'box-sizing:border-box">']
    o.append(f'<p style="margin:0 0 4px;font-size:21px;font-weight:700;color:{T_INK}">세탁 패턴 비교 리포트</p>')
    o.append(f'<p {sub}>' + " · ".join(f"{_esc(r['name'])}: {r['start_utc']} UTC 시작" for r in results) + "</p>")
    synth = [r["name"] for r in results if r.get("synthetic")]
    if synth:
        o.append(f'<p {box}><b>참고</b>: {_esc(", ".join(synth))}는 실습용 스크립트(make_sample.py)가 '
                 f'무작위로 만든 <b>합성 데이터</b>다. 주소·금액·시각은 모두 가짜이며 실제 사건이 아니다. '
                 f'실제 세탁에서 관찰되는 두 패턴(빠른 분산형, 필 체인형)을 단순화해 흉내 냈다.</p>')
    o.append(f'<p {p}>공개 블록체인 기록에서 시드(공격자) 지갑을 출발점으로 자금 흐름을 따라가, '
             f'돈이 얼마나 빨리, 얼마나 넓게 퍼져 브릿지·스왑 같은 출구로 빠져나가는지를 지표로 비교했다.</p>')

    o.append(f'<p {h}>1. 출구 도달 속도</p>')
    o.append(f'<p {sub}>가로축은 시드에서 처음 돈이 나간 뒤 흐른 시간, 세로축은 그때까지 출구(컨트랙트)에 '
             f'도달한 금액의 누적 비율이다. 선이 빨리 오를수록 세탁이 빠르고, 동결할 시간이 짧다.</p>')
    o.append(_legend(results))
    o.append(_svg_curve(results))
    o.append(f'<ul {ul}>' + "".join(f"<li {LI}>{x}</li>" for x in _speed_summary(results)) + "</ul>")

    o.append(f'<p {h}>2. 단계별 지갑 수 (분산 정도)</p>')
    o.append(f'<p {sub}>hop은 시드에서 몇 번 송금을 거쳐 도달했는지를 뜻한다. 각 단계에서 돈을 받은 개인 지갑 수가 '
             f'빨리 늘면 수많은 지갑으로 흩뿌리는 분산형이고, 단계마다 1~2개로 길게 이어지면 필 체인형이다.</p>')
    o.append(_legend(results))
    o.append(_svg_bars(results))
    o.append(f'<ul {ul}>' + "".join(f"<li {LI}>{x}</li>" for x in _spread_summary(results)) + "</ul>")

    o.append(f'<p {h}>3. 지표 표</p>')
    cell = "padding:8px;border:0;background:none;font-size:14px;line-height:1.5;vertical-align:middle;"
    th = f'style="{cell}border-bottom:2px solid {T_INK};text-align:right;font-weight:700;white-space:nowrap;color:{T_INK}"'
    th0 = f'style="{cell}border-bottom:2px solid {T_INK};text-align:left;font-weight:700;color:{T_INK}"'
    td = f'style="{cell}border-bottom:1px solid {T_GRID};text-align:right;white-space:nowrap;color:{T_INK}"'
    td0 = f'style="{cell}border-bottom:1px solid {T_GRID};text-align:left;color:{T_INK2}"'
    rows = [[label] + [fmt(r[key], f_) for r in results] for key, label, f_ in ROWS]
    for g in sorted({g for r in results for g in r["mix"]}):
        rows.append([f"자산 비중: {g}"] + [f"{r['mix'].get(g, 0):.1%}" for r in results])
    o.append('<div style="overflow-x:auto"><table style="width:100%;margin:0;border:0;border-collapse:collapse;font-size:14px;'
             f'color:{T_INK}"><thead><tr><th {th0}>지표</th>'
             + "".join(f"<th {th}>{_esc(r['name'])}</th>" for r in results) + "</tr></thead><tbody>"
             + "".join(f"<tr><td {td0}>{_esc(row[0])}</td>" + "".join(f"<td {td}>{_esc(c)}</td>" for c in row[1:])
                       + "</tr>" for row in rows)
             + "</tbody></table></div>")

    o.append(f'<p {h}>지표 읽는 법</p>')
    o.append(f'<ul {ul}>' + "".join(f"<li {LI}><b>{k}</b>: {v}</li>" for k, v in GLOSSARY) + "</ul>")
    o.append(f'<p style="margin:18px 0 0;font-size:12.5px;line-height:1.6;color:{T_MUTED}">'
             f'ETH 환산은 비교용 근사치다(스테이블코인 3,000달러≈1 ETH, BTC 래핑 1개≈30 ETH). '
             f'공개 데이터에서 얻은 단서이며 사실을 확정하는 자료가 아니다.</p>')
    o.append("</div>")
    with open(path, "w", encoding="utf-8") as f:
        f.write("<!-- 티스토리: 글쓰기 > 우측 상단 '기본모드'를 'HTML'로 바꾼 뒤 이 내용을 통째로 붙여넣기 -->\n")
        f.write("".join(o))


REPORT_HTML = r"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>세탁 패턴 비교</title>
<style>
:root{color-scheme:light;
 --surface:#fcfcfb;--panel:#f3f2ef;--text-1:#0b0b0b;--text-2:#52514e;--muted:#8a8984;
 --grid:#e4e3df;--axis:#b9b8b2;--s1:#2a78d6;--s2:#eb6834;--s3:#1baf7a;}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme="light"])){color-scheme:dark;
 --surface:#1a1a19;--panel:#242422;--text-1:#f0efec;--text-2:#c3c2b7;--muted:#8f8e88;
 --grid:#33332f;--axis:#55544f;--s1:#3987e5;--s2:#d95926;--s3:#199e70;}}
:root[data-theme="dark"]{color-scheme:dark;
 --surface:#1a1a19;--panel:#242422;--text-1:#f0efec;--text-2:#c3c2b7;--muted:#8f8e88;
 --grid:#33332f;--axis:#55544f;--s1:#3987e5;--s2:#d95926;--s3:#199e70;}
*{box-sizing:border-box}
body{margin:0;background:var(--surface);color:var(--text-1);
 font:15px/1.55 system-ui,-apple-system,"Apple SD Gothic Neo","Noto Sans KR",sans-serif}
main{max-width:920px;margin:0 auto;padding:24px 16px 48px}
h1{font-size:22px;margin:0 0 4px}h2{font-size:17px;margin:32px 0 4px}
.sub{color:var(--text-2);margin:0 0 8px;font-size:14px}
.legend{display:flex;gap:16px;flex-wrap:wrap;font-size:13px;color:var(--text-2);margin:8px 0}
.legend i{display:inline-block;width:12px;height:12px;border-radius:3px;margin-right:6px;vertical-align:-1px}
.chart{position:relative;width:100%}
svg{display:block;width:100%;height:auto;overflow:visible}
.tip{position:absolute;pointer-events:none;background:var(--panel);color:var(--text-1);
 border:1px solid var(--grid);border-radius:8px;padding:6px 10px;font-size:12px;
 white-space:nowrap;opacity:0;transition:opacity .08s;box-shadow:0 2px 8px #0002}
table{width:100%;border-collapse:collapse;font-size:14px;margin-top:8px}
th,td{padding:7px 8px;border-bottom:1px solid var(--grid);text-align:right}
th:first-child,td:first-child{text-align:left;color:var(--text-2)}
th{font-weight:600}
.wrap{overflow-x:auto}
.note{font-size:13px;color:var(--muted);margin-top:24px}
ul.fns{margin:4px 0 0;padding-left:18px;font-size:14px;color:var(--text-2)}
</style></head>
<body><main>
<h1>세탁 패턴 비교 리포트</h1>
<p class="sub" id="meta"></p>

<h2>출구 도달 속도</h2>
<p class="sub">시드 첫 유출 이후 시간에 따라, 추적 유출액 중 브릿지·스왑 등 컨트랙트에 도달한 누적 비율</p>
<div class="legend" id="lg1"></div>
<div class="chart" id="c1"><div class="tip"></div></div>

<h2>hop별 주소 수 (분산 정도)</h2>
<p class="sub">각 단계에서 자금을 받은 개인 지갑(EOA) 수. 막대가 빨리 커질수록 넓게 흩뿌리는 전략</p>
<div class="legend" id="lg2"></div>
<div class="chart" id="c2"><div class="tip"></div></div>

<h2>지표 표</h2>
<div class="wrap"><table id="tbl"></table></div>

<h2>주요 출구 함수</h2>
<div id="fns"></div>

<p class="note">ETH 환산은 정렬·비교용 근사치(스테이블코인 1 ETH≈3,000달러, BTC 래핑 1≈30 ETH)입니다. 공개 데이터 기반 단서이며 사실 확정이 아닙니다.</p>
</main>
<script>
const D=__DATA__;
const COLORS=['var(--s1)','var(--s2)','var(--s3)'];
const NS='http://www.w3.org/2000/svg';
const el=(t,a={},p)=>{const e=document.createElementNS(NS,t);for(const k in a)e.setAttribute(k,a[k]);if(p)p.appendChild(e);return e;};
document.getElementById('meta').textContent=D.cases.map(c=>`${c.name}: ${c.start} UTC 시작`).join(' · ');
function legend(id){const L=document.getElementById(id);D.cases.forEach((c,i)=>{const s=document.createElement('span');s.innerHTML=`<i style="background:${COLORS[i%3]}"></i>`;s.append(c.name);L.appendChild(s);});}
legend('lg1');legend('lg2');
function niceMax(v){if(v<=0)return 1;const p=Math.pow(10,Math.floor(Math.log10(v)));for(const m of[1,2,2.5,5,10])if(m*p>=v)return m*p;return 10*p;}

// 차트 1: 누적 출구 도달 곡선 (계단형)
(function(){
 const box=document.getElementById('c1'),tip=box.querySelector('.tip');
 const W=Math.max(320,box.clientWidth),H=W<500?240:300,m={l:44,r:64,t:12,b:36};
 const xmax=niceMax(Math.max(...D.cases.map(c=>c.curve[c.curve.length-1][0]))*1.02);
 const svg=el('svg',{viewBox:`0 0 ${W} ${H}`,role:'img','aria-label':'누적 출구 도달 비율'});box.prepend(svg);
 const X=v=>m.l+v/xmax*(W-m.l-m.r),Y=v=>H-m.b-v*(H-m.t-m.b);
 for(let i=0;i<=4;i++){const v=i/4;el('line',{x1:m.l,x2:W-m.r,y1:Y(v),y2:Y(v),stroke:'var(--grid)'},svg);
  el('text',{x:m.l-8,y:Y(v)+4,'text-anchor':'end','font-size':11,fill:'var(--text-2)'},svg).textContent=(v*100)+'%';}
 const xt=W<500?4:5;for(let i=0;i<=xt;i++){const v=xmax*i/xt;el('text',{x:X(v),y:H-m.b+18,'text-anchor':'middle','font-size':11,fill:'var(--text-2)'},svg).textContent=(+v.toFixed(1))+'h';}
 el('line',{x1:m.l,x2:W-m.r,y1:Y(0),y2:Y(0),stroke:'var(--axis)'},svg);
 el('text',{x:W-m.r,y:H-4,'text-anchor':'end','font-size':11,fill:'var(--muted)'},svg).textContent='시드 첫 유출 이후 경과 시간';
 const step=c=>{let d='',prev=null;for(const[t,v]of c.curve){d+=prev===null?`M${X(t)},${Y(v)}`:`H${X(t)}V${Y(v)}`;prev=v;}return d+`H${X(xmax)}`;};
 D.cases.forEach((c,i)=>{el('path',{d:step(c),fill:'none',stroke:COLORS[i%3],'stroke-width':2,'stroke-linejoin':'round'},svg);
  const last=c.curve[c.curve.length-1][1];
  el('text',{x:W-m.r+6,y:Y(last)+4+(i?12:0)*(Math.abs(last-(D.cases[0].curve.slice(-1)[0][1]))<.06?1:0),'font-size':12,fill:'var(--text-1)'},svg).textContent=c.name;});
 const cross=el('line',{y1:m.t,y2:H-m.b,stroke:'var(--axis)','stroke-dasharray':'3 3',opacity:0},svg);
 const dots=D.cases.map((c,i)=>el('circle',{r:4,fill:COLORS[i%3],stroke:'var(--surface)','stroke-width':2,opacity:0},svg));
 const valAt=(c,t)=>{let v=0;for(const[tt,vv]of c.curve){if(tt<=t)v=vv;else break;}return v;};
 const hit=el('rect',{x:m.l,y:m.t,width:W-m.l-m.r,height:H-m.t-m.b,fill:'transparent'},svg);
 const move=ev=>{const r=svg.getBoundingClientRect(),px=(ev.clientX-r.left)/r.width*W;
  const t=Math.max(0,Math.min(xmax,(px-m.l)/(W-m.l-m.r)*xmax));cross.setAttribute('x1',X(t));cross.setAttribute('x2',X(t));cross.setAttribute('opacity',1);
  let html=`<b>${t.toFixed(2)}시간</b>`;D.cases.forEach((c,i)=>{const v=valAt(c,t);dots[i].setAttribute('cx',X(t));dots[i].setAttribute('cy',Y(v));dots[i].setAttribute('opacity',1);
   html+=`<br><span style="color:${COLORS[i%3]}">●</span> ${c.name}: ${(v*100).toFixed(1)}%`;});
  tip.innerHTML=html;tip.style.opacity=1;const bx=box.getBoundingClientRect();let lx=ev.clientX-bx.left+12;if(lx>bx.width-170)lx-=190;tip.style.left=lx+'px';tip.style.top='8px';};
 hit.addEventListener('pointermove',move);hit.addEventListener('pointerleave',()=>{tip.style.opacity=0;cross.setAttribute('opacity',0);dots.forEach(d=>d.setAttribute('opacity',0));});
})();

// 차트 2: hop별 EOA 수 (그룹 막대)
(function(){
 const box=document.getElementById('c2'),tip=box.querySelector('.tip');
 const hops=Math.max(...D.cases.map(c=>c.per_hop.length));
 const W=Math.max(320,box.clientWidth),H=W<500?220:260,m={l:40,r:8,t:20,b:36};
 const ymax=niceMax(Math.max(...D.cases.flatMap(c=>c.per_hop.map(h=>h.eoa))));
 const svg=el('svg',{viewBox:`0 0 ${W} ${H}`,role:'img','aria-label':'hop별 EOA 수'});box.prepend(svg);
 const Y=v=>H-m.b-v/ymax*(H-m.t-m.b);
 const yt=ymax%4===0?4:5;for(let i=0;i<=yt;i++){const v=ymax*i/yt;el('line',{x1:m.l,x2:W-m.r,y1:Y(v),y2:Y(v),stroke:'var(--grid)'},svg);
  el('text',{x:m.l-8,y:Y(v)+4,'text-anchor':'end','font-size':11,fill:'var(--text-2)'},svg).textContent=Math.round(v);}
 const gw=(W-m.l-m.r)/hops,n=D.cases.length,bw=Math.min(36,(gw*0.7-2*(n-1))/n);
 for(let h=0;h<hops;h++){
  const gx=m.l+gw*h+gw/2,start=gx-(n*bw+2*(n-1))/2;
  if(gw>=34||h%Math.ceil(34/gw)===0)el('text',{x:gx,y:H-m.b+18,'text-anchor':'middle','font-size':11,fill:'var(--text-2)'},svg).textContent=gw>=44?'hop '+h:''+h;
  D.cases.forEach((c,i)=>{const v=(c.per_hop[h]||{eoa:0}).eoa,x=start+i*(bw+2),y=Y(v),hgt=H-m.b-y;
   if(hgt>0){const r=Math.min(4,hgt,bw/2);
    el('path',{d:`M${x},${H-m.b}V${y+r}Q${x},${y} ${x+r},${y}H${x+bw-r}Q${x+bw},${y} ${x+bw},${y+r}V${H-m.b}Z`,fill:COLORS[i%3]},svg);}
   const hit=el('rect',{x:x-1,y:m.t,width:bw+2,height:H-m.t-m.b,fill:'transparent'},svg);
   hit.addEventListener('pointermove',ev=>{const bx=box.getBoundingClientRect();tip.innerHTML=`<b>${c.name} · hop ${h}</b><br>EOA ${v}개 · 컨트랙트 ${(c.per_hop[h]||{contract:0}).contract}개`;
    tip.style.opacity=1;let lx=ev.clientX-bx.left+12;if(lx>bx.width-180)lx-=200;tip.style.left=lx+'px';tip.style.top='8px';});
   hit.addEventListener('pointerleave',()=>tip.style.opacity=0);});
 }
 el('line',{x1:m.l,x2:W-m.r,y1:Y(0),y2:Y(0),stroke:'var(--axis)'},svg);
})();

// 표
(function(){const T=document.getElementById('tbl');
 T.innerHTML='<thead><tr>'+D.head.map(h=>`<th>${h}</th>`).join('')+'</tr></thead><tbody>'+
 D.rows.map(r=>'<tr>'+r.map(c=>`<td>${c}</td>`).join('')+'</tr>').join('')+'</tbody>';})();
(function(){const F=document.getElementById('fns');
 F.innerHTML=D.cases.map((c,i)=>`<p style="margin:12px 0 0"><span style="color:${COLORS[i%3]}">●</span> <b>${c.name}</b></p><ul class="fns">`+
  (c.fns.length?c.fns.map(([f,v])=>`<li>${f} — 약 ${v.toLocaleString(undefined,{maximumFractionDigits:1})} ETH 환산</li>`).join(''):'<li>출구 유입 없음</li>')+'</ul>').join('');})();
</script></body></html>
"""

if __name__ == "__main__":
    main()


## 1. 합성 데이터로 두 세탁 패턴 비교

주소와 거래는 모두 무작위로 만든 **가짜**이며 실제 블록체인과 무관합니다.

- **사례 A (빠른 분산형)**: 수십 개 지갑으로 흩뿌린 뒤 몇 시간 안에 브릿지·스왑으로 빠져나감
- **사례 B (필 체인형)**: 한 지갑에서 다음 지갑으로 넘기며 매 단계 일부만 떼어 출구로 보냄

`SEED` 값을 바꾸면 다른 데이터가 만들어집니다. 학생마다 다른 값을 주면 답안 베끼기를 막을 수 있습니다.

In [ ]:
#@title 1-1. 합성 데이터 만들기
SEED = 7  #@param {type:"integer"}
!python make_sample.py --seed {SEED} --out sample

In [ ]:
#@title 1-2. 송금 기록 살펴보기 (사례 A 처음 15건)
import pandas as pd
pd.set_option("display.max_colwidth", 20)
edges_a = pd.read_csv("sample/case_A/edges.csv")
print(f"사례 A 송금 {len(edges_a)}건, 사례 B 송금 {len(pd.read_csv('sample/case_B/edges.csv'))}건")
edges_a.head(15)

In [ ]:
#@title 1-3. 두 사례 분석하고 비교 리포트 만들기
!python analyze.py 사례A=sample/case_A 사례B=sample/case_B --out report

## 2. 지표 표와 리포트 읽기

아래 셀은 지표 표와 인터랙티브 리포트를 보여 줍니다. 리포트의 출구 도달 곡선에 마우스를 올리면 시간별 값이 나옵니다.

**생각해 볼 질문**
- 사례 A의 곡선이 가파르게 오르는 구간은 언제이고, 그때 무슨 일이 일어나는가?
- 사례 B의 곡선은 왜 계단 모양인가?
- 거래소가 탈취 6시간 뒤에 주소 목록을 공유했다면, 각 사례에서 아직 출구를 지나지 않은 비율은 얼마인가?

In [ ]:
#@title 2-1. 지표 표
import pandas as pd
pd.read_csv("report/metrics.csv")

In [ ]:
#@title 2-2. 비교 리포트 (차트)
from IPython.display import HTML, display
display(HTML(open("report/report.html", encoding="utf-8").read()))

### 블로그(티스토리)에 올리기

`analyze.py`는 리포트를 두 가지로 만듭니다.

| 파일 | 용도 |
| --- | --- |
| `report.html` | 브라우저로 여는 인터랙티브 리포트 (마우스를 올리면 값 표시) |
| `report_tistory.html` | 블로그 본문에 붙여 넣는 버전. 해설 문장, 정적 차트, 지표 표, 용어 설명 포함 |

블로그용은 스크립트 없이 그림(SVG)으로 차트를 그리고 스타일을 모두 태그 안에 넣어서, 블로그 테마와 섞이거나 깨지지 않습니다.
합성 데이터로 만든 사례에는 "실제 사건이 아니다"라는 안내 상자가 자동으로 붙습니다.

**올리는 순서**
1. 아래 셀을 실행해 `report_tistory.html`을 내려받습니다. (미리보기도 함께 표시됩니다.)
2. 파일을 메모장(Windows) 또는 텍스트 편집기(Mac)로 열어 내용을 **전부 복사**합니다. 브라우저로 열면 안 됩니다.
3. 티스토리 글쓰기 화면 오른쪽 위의 **기본모드**를 눌러 **HTML**로 바꿉니다.
4. 원하는 위치에 붙여 넣고, 다시 **기본모드**로 돌아와 모양을 확인한 뒤 발행합니다.

> 기본모드에서 리포트 안쪽 글자를 고치면 스타일이 풀릴 수 있습니다. 수정은 HTML 모드에서 하세요.

In [ ]:
#@title 2-3. 블로그용 리포트 미리보기·내려받기
from IPython.display import HTML, display
frag = open("report/report_tistory.html", encoding="utf-8").read()
display(HTML(frag))
try:
    from google.colab import files
    files.download("report/report_tistory.html")
except ImportError:
    pass

## 3. (선택) 실제 공개 주소로 추적

같은 분석을 수사기관이 공개한 실제 주소로 반복합니다.

**준비: Etherscan API 키 등록 (처음 한 번만)**
1. [etherscan.io/myapikey](https://etherscan.io/myapikey)에서 무료 회원가입 후 API 키를 발급받습니다.
2. Colab 왼쪽 사이드바의 **열쇠(🔑) 아이콘**을 누릅니다.
3. **새 보안 비밀 추가**를 누르고, 이름에 `ETHERSCAN_API_KEY`, 값에 발급받은 키를 넣습니다.
4. **노트북 액세스**를 켭니다.

> API 키를 코드 셀에 직접 적지 마세요. 노트북을 공유하면 키도 함께 공개됩니다.

In [ ]:
#@title 3-1. API 키 불러오기
import os
try:
    from google.colab import userdata
    os.environ["ETHERSCAN_API_KEY"] = userdata.get("ETHERSCAN_API_KEY")
    print("✅ API 키를 불러왔습니다.")
except Exception as e:
    print("❌ API 키를 불러오지 못했습니다:", type(e).__name__)
    print("   왼쪽 🔑 보안 비밀에 ETHERSCAN_API_KEY를 등록하고 '노트북 액세스'를 켰는지 확인하세요.")

**시드 주소 입력**: FBI 등이 공개한 ETH 주소를 아래 셀의 따옴표 사이에 한 줄에 하나씩 붙여 넣으세요.
BTC·SOL 주소나 `#` 뒤의 주석은 자동으로 건너뜁니다.

파일로 올리고 싶으면 `UPLOAD = True`로 바꾸고 실행하세요. 업로드한 파일이 `addresses.txt`로 저장됩니다.

In [ ]:
#@title 3-2. 시드 주소 입력
UPLOAD = True  #@param {type:"boolean"}

ADDRESSES = """
# 예) 0x0000000000000000000000000000000000000000
"""

if UPLOAD:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    open("addresses.txt", "wb").write(up[name])
else:
    open("addresses.txt", "w", encoding="utf-8").write(ADDRESSES)

seeds = [l.split("#")[0].strip().strip(",") for l in open("addresses.txt", encoding="utf-8")]
seeds = [s for s in seeds if s.lower().startswith("0x") and len(s) == 42]
print(f"ETH 시드 주소 {len(seeds)}개를 읽었습니다.")
if not seeds:
    print("⚠️ 주소가 없습니다. ADDRESSES에 주소를 붙여 넣고 다시 실행하세요.")

**추적 실행**

- `SINCE_BLOCK`: 해킹 직전 블록 번호. Etherscan에서 시드 주소의 첫 유입 거래를 열면 나옵니다. 0이면 과거 기록이 모두 섞입니다.
- `HOPS`: 시드에서 몇 단계까지 따라갈지. 처음에는 2로 시작하세요.
- `MAX_NODES`: 시드 외에 새로 추적할 최대 주소 수. 무료 API는 초당 5회 제한이라 주소 하나에 약 1초가 걸립니다.
  시드 600개 + 새 주소 300개면 약 15분입니다. 실행하면 예상 시간이 먼저 표시됩니다.

In [ ]:
#@title 3-3. 추적 실행
SINCE_BLOCK = 0  #@param {type:"integer"}
HOPS = 1  #@param {type:"slider", min:1, max:4, step:1}
MAX_NODES = 300  #@param {type:"integer"}
OUT = "out_real"  #@param {type:"string"}
!python trace_flows.py addresses.txt --since-block {SINCE_BLOCK} --hops {HOPS} --max-nodes {MAX_NODES} --out {OUT}

In [ ]:
import pandas as pd
e = pd.read_csv("out_real/edges.csv")
n = pd.read_csv("out_real/nodes.csv")
display(e[["from", "to", "asset", "amount", "time_utc", "etherscan"]])
display(n[n.hop > 0][["address", "type", "in_eth_equiv", "etherscan"]])

In [ ]:
print(e["to"].tolist())

In [ ]:
#@title 3-4. 출구 후보 (유입액 상위 컨트랙트)
import pandas as pd, os
path = f"{OUT}/exits.csv"
if os.path.exists(path):
    display(pd.read_csv(path).head(20))
    print("Etherscan 링크를 열어 컨트랙트 이름표(라벨)를 확인해 보세요.")
else:
    print("결과가 없습니다. 3-3 셀이 정상적으로 끝났는지 확인하세요.")

In [ ]:
import pandas as pd
print("결과 폴더:", OUT)
n = pd.read_csv(f"{OUT}/nodes.csv"); e = pd.read_csv(f"{OUT}/edges.csv")
print(f"주소 {len(n)}개, 송금 {len(e)}건")
display(n[["address", "hop", "type", "in_eth_equiv", "out_eth_equiv"]])
display(e[["from", "to", "amount", "time_utc"]])

In [ ]:
#@title 3-5. 흐름 그래프 (노드 더블클릭 시 Etherscan)
from IPython.display import HTML, display
import os
path = f"{OUT}/graph.html"
display(HTML(open(path, encoding="utf-8").read()) if os.path.exists(path) else "결과가 없습니다.")

In [ ]:
#@title 3-6. 합성 사례와 나란히 비교
NAME = "실제사건"  #@param {type:"string"}
!python analyze.py 사례A=sample/case_A 사례B=sample/case_B {NAME}={OUT} --out report_real
from IPython.display import HTML, display
import os
if os.path.exists("report_real/report.html"):
    display(HTML(open("report_real/report.html", encoding="utf-8").read()))

In [ ]:
#@title 4. 모니터링 시트 자동 채우기 (잔액·보낸 거래 수)
# 준비: 3-1 셀(API 키)을 먼저 실행하세요.
# 실행하면 binance_wallet_monitor.xlsx 업로드 창이 뜹니다.
# 608개 주소의 잔액과 보낸 거래 수를 조회해 시트를 채우고, 조회기록에 한 줄을 추가한 뒤 내려받습니다.
import os, sys, json, time, urllib.request, urllib.parse
from datetime import date, datetime
from openpyxl import load_workbook

API = "https://api.etherscan.io/v2/api"
RPS = 3.0                     # 무료 키는 초당 3회 권장
XLSX = "binance_wallet_monitor.xlsx"
SHEET = "주소현황"
DUST = 0.01                   # 이 값(ETH)보다 적게 남으면 '전액 이동'으로 본다

KEY = os.environ.get("ETHERSCAN_API_KEY", "").strip()
if not KEY:
    raise SystemExit("API 키가 없습니다. 3-1 셀을 먼저 실행하세요.")

_last = [0.0]
def call(**params):
    """Etherscan 호출. 일시적 거부·한도 초과는 쉬었다가 재시도한다."""
    params.update(chainid=1, apikey=KEY)
    url = API + "?" + urllib.parse.urlencode(params)
    for attempt in range(6):
        wait = 1.0 / RPS - (time.time() - _last[0])
        if wait > 0:
            time.sleep(wait)
        _last[0] = time.time()
        try:
            data = json.loads(urllib.request.urlopen(url, timeout=30).read().decode())
        except Exception as e:
            print(f"  ! 요청 오류({e}), 재시도"); time.sleep(2 * (attempt + 1)); continue
        res = data.get("result")
        if isinstance(res, str) and ("rate limit" in res.lower() or "Invalid API Key" in res):
            time.sleep(5 * (attempt + 1)); continue
        if data.get("status") == "0" and "jsonrpc" not in data:
            print("  ! API 응답:", data.get("message"), str(res)[:80]); return None
        return res
    print("  ! 재시도 한도 초과"); return None

# 1) 시트 업로드
if not os.path.exists(XLSX):
    from google.colab import files
    print(f"{XLSX} 파일을 선택하세요.")
    up = files.upload()
    name = next(iter(up)); open(XLSX, "wb").write(up[name])
wb = load_workbook(XLSX)
ws = wb[SHEET]
rows = [r for r in range(2, ws.max_row + 1)
        if isinstance(ws.cell(r, 3).value, str) and ws.cell(r, 3).value.lower().startswith("0x")]
addrs = [ws.cell(r, 3).value for r in rows]
print(f"주소 {len(addrs)}개를 조회합니다. 예상 시간 약 {(len(addrs) + len(addrs) / 20) / RPS / 60:.0f}분")

# 2) 잔액: 한 번에 20개씩
bal = {}
for i in range(0, len(addrs), 20):
    chunk = addrs[i:i + 20]
    res = call(module="account", action="balancemulti", address=",".join(chunk), tag="latest") or []
    for item in res:
        bal[item["account"].lower()] = int(item["balance"]) / 1e18
print(f"잔액 조회 완료: {len(bal)}/{len(addrs)}")

# 3) 보낸 거래 수(nonce): 주소마다 1번
sent = {}
for n, a in enumerate(addrs, 1):
    res = call(module="proxy", action="eth_getTransactionCount", address=a, tag="latest")
    if isinstance(res, str) and res.startswith("0x"):
        sent[a.lower()] = int(res, 16)
    if n % 50 == 0:
        print(f"  보낸 거래 수 조회 {n}/{len(addrs)}")
print(f"보낸 거래 수 조회 완료: {len(sent)}/{len(addrs)}")

# 4) 시트에 기록 (라벨·상위 지갑·메모 열은 그대로 둔다)
today = date.today()
for r, a in zip(rows, addrs):
    k = a.lower()
    if k not in bal or k not in sent:
        continue                      # 조회 실패한 주소는 이전 값 유지
    b, s = bal[k], sent[k]
    status = "미이동" if s == 0 else ("전액 이동" if b < DUST else "이동 시작")
    ws.cell(r, 6, b); ws.cell(r, 7, s); ws.cell(r, 9, today); ws.cell(r, 10, status)

# 5) ETH 시세와 조회기록 한 줄
price = call(module="stats", action="ethprice")
eth_usd = float(price["ethusd"]) if isinstance(price, dict) else None
if eth_usd:
    wb["요약"]["B3"] = eth_usd
done = [(bal[a.lower()], sent[a.lower()]) for a in addrs if a.lower() in bal and a.lower() in sent]
n_still = sum(1 for b, s in done if s == 0)
n_moving = sum(1 for b, s in done if s > 0 and b >= DUST)
total = sum(b for b, s in done)
log = wb["조회기록"]
r = 2
while log.cell(r, 1).value not in (None, ""):
    r += 1
log.cell(r, 1, today); log.cell(r, 2, len(done)); log.cell(r, 3, n_still); log.cell(r, 4, n_moving)
log.cell(r, 5, total); log.cell(r, 6, eth_usd)
log.cell(r, 8, f"Colab 자동 조회 {datetime.now():%H:%M} UTC")

wb.save(XLSX)

# 6) 결과 요약
print("\n===== 결과 =====")
print(f"확인한 주소   {len(done)} / {len(addrs)}")
print(f"미이동        {n_still}개 (보낸 거래 0건)")
print(f"이동 시작     {n_moving}개")
print(f"전액 이동     {len(done) - n_still - n_moving}개")
print(f"남은 잔액     {total:,.2f} ETH" + (f"  (약 {total * eth_usd / 1e8:,.1f}억 달러, 시세 {eth_usd:,.2f})" if eth_usd else ""))
moved = [(a, bal[a.lower()], sent[a.lower()]) for a in addrs
         if a.lower() in sent and sent[a.lower()] > 0]
if moved:
    print("\n송금 기록이 있는 주소 (먼저 살펴볼 곳):")
    for a, b, s in sorted(moved, key=lambda x: -x[2])[:15]:
        print(f"  {a}  보낸 거래 {s}건, 잔액 {b:,.2f} ETH")

try:
    from google.colab import files
    files.download(XLSX)
except ImportError:
    print(f"\n{XLSX} 저장 완료")

## 결과 내려받기

아래 셀을 실행하면 지금까지 만든 결과 폴더를 `results.zip` 하나로 묶어 내려받습니다.
리포트(`report.html`, `graph.html`)는 내려받은 뒤 브라우저로 열 수 있습니다.

In [ ]:
#@title 결과 묶어서 내려받기
import os, shutil, glob
folders = [d for d in ["sample", "report", "report_real"] + glob.glob("out_*") if os.path.isdir(d)]
os.makedirs("_results", exist_ok=True)
for d in folders:
    shutil.copytree(d, f"_results/{d}", dirs_exist_ok=True)
shutil.make_archive("results", "zip", "_results")
print("묶은 폴더:", ", ".join(folders))
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Colab이 아니어서 results.zip 파일만 만들었습니다.")

---
**연구 윤리**: 분석 결과는 단서일 뿐 사실 확정이 아닙니다. 특정 개인·기업을 공개적으로 지목하지 말고,
의미 있는 발견은 트랜잭션 해시를 근거로 [tips.fbi.gov](https://tips.fbi.gov) 또는 국내 수사기관에 제보하세요.